# Тема 6. Модель похибок мовою Python

Ви відкрили Jupyter Notebook: файл, у якому пояснення чергуються з короткими програмами мовою Python. Програми цього файлу багато разів «вимірюють» пеленги постів із випадковою похибкою і показують, де щоразу опиняється знайдене місце передавача. Пости й передавач стоять так само, як на рис. 6.1 теми 6.

**Як працювати.** Jupyter Notebook складається з комірок: одні комірки містять текст, як ця, а інші програму. Клацніть першу комірку з програмою (вона розташована одразу під цим текстом) і натисніть Shift+Enter: комірка виконається, а під нею з’явиться результат. Так само виконуйте решту комірок по черзі, згори донизу.

In [ ]:
# Пости й передавач. Координати в кілометрах: x на схід, y на північ.
import matplotlib.pyplot as plt
import numpy as np

POSTS = {"А": (0.0, 0.0), "Б": (25.0, 0.0), "В": (21.0, 24.0)}
EMITTER = (16.0, 12.0)


def uk(number, digits=1):
    """Число з десятковою комою."""
    return f"{number:.{digits}f}".replace(".", ",")


def bearing(post, point):
    """Пеленг із поста на точку: градуси від півночі за годинниковою стрілкою."""
    return np.degrees(np.arctan2(point[0] - post[0], point[1] - post[1])) % 360


def crossing(post_a, bearing_a, post_b, bearing_b):
    """Точка, у якій перетинаються лінії двох пеленгів."""
    a, b = np.radians(bearing_a), np.radians(bearing_b)
    far = ((post_b[0] - post_a[0]) * np.cos(b) - (post_b[1] - post_a[1]) * np.sin(b)) / np.sin(a - b)
    return post_a[0] + far * np.sin(a), post_a[1] + far * np.cos(a)


for name, post in POSTS.items():
    print(f"Пост {name}: пеленг {uk(bearing(post, EMITTER))}°")

## Крок 1. Одне вимірювання з похибкою

Справжній пост вимірює пеленг із похибкою. Програма додає до кожного пеленга випадкове число від `-ERROR` до `ERROR` градусів. Місце передавача програма шукає так, як у темі 6: два пости дають точку перетину пеленгів, три пости дають три точки, і місцем передавача вважають їхній центр.

Промах — це відстань від знайденого місця до справжнього місця передавача.

In [ ]:
ERROR = 3.0                  # найбільша похибка пеленга, градуси
USED = ["А", "Б", "В"]            # пости, які вимірюють пеленги
rng = np.random.default_rng(1)    # джерело випадкових чисел


def estimate():
    """Одне вимірювання: місце передавача, знайдене за пеленгами з випадковою похибкою."""
    measured = {name: bearing(POSTS[name], EMITTER) + rng.uniform(-ERROR, ERROR) for name in USED}
    points = [crossing(POSTS[a], measured[a], POSTS[b], measured[b])
              for i, a in enumerate(USED) for b in USED[i + 1:]]
    return np.mean(points, axis=0)


x, y = estimate()
print(f"Знайдене місце: x = {uk(x)} км, y = {uk(y)} км")
print(f"Промах цього вимірювання: {uk(np.hypot(x - EMITTER[0], y - EMITTER[1]), 2)} км")

## Крок 2. Сотні вимірювань

За одним вимірюванням точність оцінити не можна: похибка щоразу інша. Наступна програма повторює вимірювання 500 разів і позначає кожне знайдене місце точкою. Під рисунком програма друкує середній і найбільший промах.

In [ ]:
TRIALS = 500
rng = np.random.default_rng(1)
found = np.array([estimate() for _ in range(TRIALS)])
miss = np.hypot(found[:, 0] - EMITTER[0], found[:, 1] - EMITTER[1])

figure, (scene, near) = plt.subplots(1, 2, figsize=(12, 5.5))
for name in USED:                     # ліворуч: пости та лінії їхніх пеленгів без похибки
    scene.plot([POSTS[name][0], EMITTER[0]], [POSTS[name][1], EMITTER[1]], color="tab:blue", linewidth=0.8)
    scene.scatter(*POSTS[name], marker="s", s=60, color="tab:blue")
    scene.annotate("пост " + name, POSTS[name], textcoords="offset points", xytext=(8, 8))
for picture in (scene, near):
    picture.scatter(found[:, 0], found[:, 1], s=6, color="tab:red", label="знайдені місця")
    picture.scatter(*EMITTER, marker="x", s=80, color="black", label="передавач")
    picture.set_aspect("equal", adjustable="datalim")
    picture.grid(alpha=0.3)
    picture.set_xlabel("x, км")
    picture.set_ylabel("y, км")
span = max(2.0, 1.1 * miss.max())     # праворуч: ті самі точки зблизька
near.set_xlim(EMITTER[0] - span, EMITTER[0] + span)
near.set_ylim(EMITTER[1] - span, EMITTER[1] + span)
scene.set_title(f"{TRIALS} вимірювань, похибка пеленга до {uk(ERROR, 0)}°")
near.set_title("Знайдені місця зблизька")
near.legend()
plt.show()

print(f"Середній промах: {uk(miss.mean(), 2)} км")
print(f"Найбільший промах: {uk(miss.max(), 2)} км")

## Завдання

Після кожної зміни виконуйте програми знову, згори донизу.

1. Запишіть середній промах. Замініть рядок `ERROR = 3.0` на `ERROR = 1.0`, потім на `ERROR = 6.0`. У скільки разів змінився середній промах? Поверніть `ERROR = 3.0`.
2. Залиште два пости: замініть рядок `USED = ["А", "Б", "В"]` на `USED = ["А", "Б"]`. Запишіть середній промах. Потім поставте `USED = ["А", "В"]`. Яка пара точніша? Подивіться на рисунок: під яким кутом перетинаються пеленги кожної пари?
3. Поверніть `USED = ["А", "Б"]`. У першій програмі перенесіть пост Б ближче до поста А: замініть `"Б": (25.0, 0.0)` на `"Б": (6.0, 0.0)`. Що сталося з хмаркою знайдених місць і з промахом? Тепер поверніть третій пост: `USED = ["А", "Б", "В"]`. Чи допоміг він?
4. У короткому звіті запишіть, як на точність впливають похибка пеленга й розташування постів. Де ви поставили б третій пост, якщо два перші стоять поруч?